# Demo day: your six minutes, in one notebook

Run this top to bottom on Friday, one cell per beat. Every cell is a command from
`docs/DEFENCE.md`, and every one ends in a **receipt** or a **refusal**.

| Clock | Beat | Cell |
|---|---|---|
| 0:00 | Your README: one sentence and the explorer link | 1 |
| 0:45 | Gecko lists your store | 2 |
| 1:30 | The live buy | 3 |
| 2:30 | The landing: the receipt, read from the ledger | 4 |
| 3:15 | **The card**: the judge draws one | 5 |
| 4:30 | Tests and the five cases | 6 |
| 5:15 | The ADR | 7 |
| finalists | Mainnet: `show`, one espresso, `show` | 8 |
| any time | The way back: the recorded lane | 9 |

Open it with  from your repository.
Run it once on Thursday against the clock, and keep the outputs: a notebook that
already ran is your fallback if the network fails on stage. Say so if you use it.

In [ ]:
# Setup. Works from anywhere inside your my-gecko-buyer repository.
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
STORE = json.loads((ROOT / "store" / "store.json").read_text())["store"]


def run(command, tail=30, env=None):
    """Run one command from the repository root and show what it printed."""
    print(f"$ {command}")
    clean = {k: v for k, v in os.environ.items() if k != "VIRTUAL_ENV"}
    done = subprocess.run(
        command,
        shell=True,
        cwd=ROOT,
        text=True,
        capture_output=True,
        env={**clean, "NO_COLOR": "1", **(env or {})},
    )
    lines = (done.stdout + done.stderr).rstrip().splitlines()
    print("\n".join(lines if len(lines) <= tail else ["..."] + lines[-tail:]))
    return done


def newest(folder, pattern):
    files = sorted((ROOT / folder).glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


print("repository:", ROOT.name, "| store:", STORE)

repository: my-gecko-buyer | store: dev3messibre


## 1 · 0:00 · Your README

One sentence, then the explorer link to your landed devnet purchase.

In [ ]:
print("\n".join((ROOT / "README.md").read_text().splitlines()[:8]))

# My Gecko Buyer Agent

Open your own store on Solana devnet and build a buyer agent that buys from it through Gecko: it pins what was asked before any bytes exist, refuses by field when the prepared purchase disagrees, signs only after a passing simulation, and writes one receipt, read from the ledger, that says what moved.

**Devnet Explorer Link:** https://explorer.solana.com/tx/<YOUR_LANDED_SIGNATURE>?cluster=devnet

## The Landing (Reconciled Receipt)



## 2 · 0:45 · Gecko lists your store

On stage, show it from your assistant with Gecko connected. This cell is the same call,
`list_stores`, through Gecko's keyless hosted MCP.

In [3]:
from buyer.mcp_client import HostedGecko

answer = HostedGecko().call("list_stores", {"store": STORE, "network": "devnet"})
for store in answer.get("stores", []):
    if store.get("store") == STORE:
        print(
            f"{store['store']} at {store['address']}, "
            f"total_purchases {store.get('total_purchases')}"
        )
        for product in store["products"]:
            print(
                f"  {product['name']:<12} {product['price_raw']:>9} raw  "
                f"mint {product['mint'][:8]}.."
            )

dev3messibre at E2TFDVMiYjPQGwGBkqcAz2tb96cFKVB6Y9tHZym9PZpY, total_purchases 3
  Espresso       1000000 raw  mint 2JqTKJBQ..
  Cookie         2500000 raw  mint 2JqTKJBQ..
  Beans          4000000 raw  mint 2JqTKJBQ..


## 3 · 1:30 · The live buy

Pin, prepare, seven checks, sign, verify, submit. Read the ticks out loud.

In [4]:
buy = run('uv run buyer "one espresso" --devnet')

$ uv run buyer "one espresso" --devnet
dev3messibre  "one espresso"  (devnet)
  [  ok] signer    devnet 9pvjyoupV6ye7QL9KfKQdL781VPE9zDAhRU8jPB5JP9d (genesis checked before each signature)
  [  ok] menu      dev3messibre: 3 products, E2TFDVMiYjPQGwGBkqcAz2tb96cFKVB6Y9tHZym9PZpY
  [  ok] pin       1 x 'Espresso', budget 2000000, mint 2JqTKJBQ..  -> intents\20261002T125632.265078-one-espresso.json
  [  ok] prepare   simulated: pass, expires in 150 blocks
  [  ok] check     program: 'BUYuxRfhCMWavaUWxhGtPP3ksKEDZxCD5gzknk3JfAya'
  [  ok] check     store: 'E2TFDVMiYjPQGwGBkqcAz2tb96cFKVB6Y9tHZym9PZpY'
  [  ok] check     product: 'Espresso'
  [  ok] check     price_raw: 1000000
  [  ok] check     mint: '2JqTKJBQCtavKdTBALyTiBL69PkCyKrRwb6q65k6deLo'
  [  ok] check     quantity: 1
  [  ok] check     destination: 'BwFetuZuMBvLCeTMV7MBH5y6niTcd6nEuyYseiyhWe2Y'
  [  ok] sign      signed by 9pvjyoupV6ye7QL9KfKQdL781VPE9zDAhRU8jPB5JP9d
  [  ok] verify    the signed bytes are the prepared bytes
  [

## 4 · 2:30 · The landing

The receipt the buyer wrote: two ledger reads, the deltas, `total_purchases` from n to n+1,
and the explorer link. Open the link.

In [5]:
receipt = newest("receipts", "*.md")
print(receipt.relative_to(ROOT) if receipt else "no receipt yet")
print(receipt.read_text() if receipt else "")

receipts\5B5Eq9Yt.md
# Receipt 5B5Eq9Yt

**one espresso**, from `dev3messibre` on devnet: reconciled with the ledger.

- signature: `5B5Eq9YtRR8AGCzVUG2GZvhki4NL4qfm9kUDKiRYkkFxVTbTXcxBUMsr53exW2BCBuMHUBtQ7tAGT8MTPyzyyFdC`
- explorer: https://explorer.solana.com/tx/5B5Eq9YtRR8AGCzVUG2GZvhki4NL4qfm9kUDKiRYkkFxVTbTXcxBUMsr53exW2BCBuMHUBtQ7tAGT8MTPyzyyFdC?cluster=devnet
- slot: 506644094
- product: Espresso
- price_raw: 1000000 of mint `2JqTKJBQCtavKdTBALyTiBL69PkCyKrRwb6q65k6deLo`
- buyer delta: -1000000
- store delta: +1000000
- total_purchases: 3 to 4
- intent pinned at: 2026-10-02T12:56:32.265078+00:00
- source: HfRurfnAGS2Lh3g4yp3ckkUKaHiJ1vqpFgrcy3MDdRmR



## 5 · 3:15 · The card

The judge draws one face down and reads it out. Set `CARD`, run the cell, and narrate.
Every card runs on **devnet**, even for finalists. Stale takes about 40 seconds: say what
it is waiting for.

In [6]:
CARD = "quantity"  # <- the judge's card: quantity, budget, tampered or stale

menu = HostedGecko().call("list_stores", {"store": STORE, "network": "devnet"})
espresso = next(
    p
    for s in menu["stores"]
    if s["store"] == STORE
    for p in s["products"]
    if p["name"].lower() == "espresso"
)
COMMANDS = {
    "quantity": 'uv run buyer "two espressos" --devnet',
    "budget": f'uv run buyer "one espresso" --budget-raw {espresso["price_raw"] // 2} --devnet',
    "tampered": 'uv run buyer "one espresso" --devnet --card tampered',
    "stale": 'uv run buyer "one espresso" --devnet --card stale',
}
card = run(COMMANDS[CARD])
refusal = newest("refusals", "*.json")
if refusal:
    print("\n", refusal.relative_to(ROOT))
    print(refusal.read_text())

$ uv run buyer "two espressos" --devnet
dev3messibre  "two espressos"  (devnet)
  [  ok] signer    devnet 9pvjyoupV6ye7QL9KfKQdL781VPE9zDAhRU8jPB5JP9d (genesis checked before each signature)
  [  ok] menu      dev3messibre: 3 products, E2TFDVMiYjPQGwGBkqcAz2tb96cFKVB6Y9tHZym9PZpY
  [  ok] pin       2 x 'Espresso', budget 2000000, mint 2JqTKJBQ..  -> intents\20261002T125828.146673-two-espressos.json
  [  ok] prepare   simulated: pass, expires in 150 blocks
  [  ok] check     program: 'BUYuxRfhCMWavaUWxhGtPP3ksKEDZxCD5gzknk3JfAya'
  [  ok] check     store: 'E2TFDVMiYjPQGwGBkqcAz2tb96cFKVB6Y9tHZym9PZpY'
  [  ok] check     product: 'Espresso'
  [  ok] check     price_raw: 1000000
  [  ok] check     mint: '2JqTKJBQCtavKdTBALyTiBL69PkCyKrRwb6q65k6deLo'
  [  NO] check     REFUSED on quantity: asked 2, prepared 1
  Nothing was signed.

 refusals\20261002T125830.113553-quantity.json
{
  "ask": "two espressos",
  "step": "check",
  "field": "quantity",
  "asked": 2,
  "found": 1,
  "where": "pre

### Thursday: rehearse all four cards

Not on stage: the night before. Set `REHEARSE = True` and run it once; you should see four
refusals and nothing signed. Stale waits about 40 seconds.

In [7]:
REHEARSE = False  # <- True on Thursday

if REHEARSE:
    for name, command in COMMANDS.items():
        print(f"--- {name}")
        out = run(command, tail=3)
else:
    print("rehearsal skipped: set REHEARSE = True the night before.")

rehearsal skipped: set REHEARSE = True the night before.


## 6 · 4:30 · Tests and the five cases

The tests trigger every refusal offline. Then the five cases and the trap, on recorded
answers. Point at one test that was red first.

In [8]:
run("uv run pytest -q -o addopts='' --color=no", tail=1)
run("uv run buyer --cases --recorded", tail=1)
run("uv run buyer --cards --recorded", tail=1);

$ uv run pytest -q -o addopts='' --color=no
...
25 failed, 73 passed, 2 skipped in 4.83s
$ uv run buyer --cases --recorded
...
6/6 cases match what the fixtures expect
$ uv run buyer --cards --recorded
...
4/4 cards match what the fixtures expect


## 7 · 5:15 · The ADR

The decision, and what would reverse it.

In [9]:
print(
    "\n".join(
        (ROOT / "docs" / "adr" / "0001-refusals-before-signing.md").read_text().splitlines()[:30]
    )
)

# The buyer signs only when 7 fields match the pinned intent

## Status and date

accepted, 2026-10-02

## Context

My buyer holds a key that can pay. Gecko prepares the bytes; I sign them. What would a wrong transaction cost, and which past incident shows it?
A wrong transaction could cost the buyer their entire wallet balance or their defined maximum cap (e.g., 2,000,000 raw units of a token). The "VIP ticket" and "Latte" test cases demonstrated that blindly trusting a server's unverified transaction bytes allows malicious actors to execute a bait-and-switchâ€”charging the correct amount but delivering an unwanted or worthless digital asset.

## Decision

Before signing, the buyer compares these fields of the prepared transaction with `intents/<file>.json` and refuses on the first mismatch, naming the field and both values:

| Field        | Compared how                                                 | Why this one                                                                     

## 8 · Finalists only: mainnet

Your own wallet, registered and funded with three espressos. `show`, the buy, `show`
again: the difference is the price, exactly. **Real money.** Leave `RUN_MAINNET = False`
unless you are a finalist with a funded wallet.

In [10]:
RUN_MAINNET = False  # <- finalists: True, on stage

if RUN_MAINNET:
    run("uv run python scripts/mainnet_wallet.py show")
    run('uv run buyer "one espresso" --mainnet --store geckocoffee')
    run("uv run python scripts/mainnet_wallet.py show")
else:
    print("mainnet skipped: RUN_MAINNET is False. Everyone else presents on devnet.")

mainnet skipped: RUN_MAINNET is False. Everyone else presents on devnet.


## 9 · The way back: the recorded lane

If devnet or Gecko is down at 1:30, run this and say so: same code path, real answers we
recorded. Presenting a replay as live is the one thing that fails.

In [11]:
run('uv run buyer "one espresso" --devnet', env={"GECKO_SOURCE": "recorded"}, tail=18);

$ uv run buyer "one espresso" --devnet
dev3pack-cafe  "one espresso"  (recorded)
  [  ok] signer    devnet E4S9vud2r3uTXKuMra7MSAe4Admop8eewMYEPLSDK5pg (recorded, no key)
  [  ok] menu      dev3pack-cafe: 6 products, AzJW94Hpu8wnNpQ9DyCvyann24tmdDKhfdKn9GxFYX5f
  [  ok] pin       1 x 'Espresso', budget 2000000, mint Eoqdd43n..  -> intents\20261002T125929.940680-one-espresso.json
  [  ok] prepare   simulated: pass, expires in 150 blocks
  [  ok] check     program: 'BUYuxRfhCMWavaUWxhGtPP3ksKEDZxCD5gzknk3JfAya'
  [  ok] check     store: 'AzJW94Hpu8wnNpQ9DyCvyann24tmdDKhfdKn9GxFYX5f'
  [  ok] check     product: 'Espresso'
  [  ok] check     price_raw: 1000000
  [  ok] check     mint: 'Eoqdd43nFQ9HzGq8HjBRVLCV6aTqCFRiwHy1ZVQheYSi'
  [  ok] check     quantity: 1
  [  ok] check     destination: '65mWSJb1xkg9GhXcker6ERJAddAYRp48QY5jKyGDughM'
  [  ok] sign      signed by E4S9vud2r3uTXKuMra7MSAe4Admop8eewMYEPLSDK5pg
  [  ok] verify    the signed bytes are the prepared bytes
  [  ok] submit    4